# Project Dataset To Regular Mesh

Notebook minimal pour projeter un dataset existant `base.bin + *.pkl` sur un maillage regulier deja cree.


In [ ]:
from pathlib import Path
import os
import pickle
import sys

import numpy as np
from scipy.spatial import Delaunay, cKDTree

import dgl
import torch


def find_project_root(start: Path) -> Path:
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "python" / "create_dgl_dataset.py").exists():
            return path
    raise RuntimeError("Cannot find gnn_modulus_test project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from python.python_code.data_manip.extraction.telemac_file import TelemacFile
from python.create_dgl_dataset import add_mesh_info, get_dgl_graph, unpack_dynamic_sample

os.chdir(PROJECT_ROOT)
PROJECT_ROOT


In [ ]:
# A remplir.
# Source attendue : un mesh fin `.slf`, un graphe `*_base.bin` et des fichiers dynamiques `.pkl`.
FINE_MESH_SLF = ""
FINE_BASE_BIN = ""
FINE_DYNAMIC_FILES = [
    "",
]
REGULAR_MESH_SLF = ""
OUTPUT_DIR = ""
DATASET_NAME = "regular_dataset"
OVERWRITE = True

print({
    "FINE_MESH_SLF": FINE_MESH_SLF,
    "FINE_BASE_BIN": FINE_BASE_BIN,
    "FINE_DYNAMIC_FILES": FINE_DYNAMIC_FILES,
    "REGULAR_MESH_SLF": REGULAR_MESH_SLF,
    "OUTPUT_DIR": OUTPUT_DIR,
    "DATASET_NAME": DATASET_NAME,
    "OVERWRITE": OVERWRITE,
})


In [ ]:
def require_path(path_like: str) -> Path:
    path = Path(path_like).expanduser()
    if not path.exists():
        raise FileNotFoundError(path)
    return path


def build_interpolator(src_xy: np.ndarray, dst_xy: np.ndarray):
    tri = Delaunay(src_xy)
    simplex = tri.find_simplex(dst_xy)
    valid = simplex >= 0

    vertices = np.zeros((valid.sum(), 3), dtype=np.int64)
    barycentric = np.zeros((valid.sum(), 3), dtype=np.float32)

    if valid.any():
        transform = tri.transform[simplex[valid], :2]
        offset = tri.transform[simplex[valid], 2]
        delta = dst_xy[valid] - offset
        barycentric[:, :2] = np.einsum("ijk,ik->ij", transform, delta)
        barycentric[:, 2] = 1.0 - barycentric[:, 0] - barycentric[:, 1]
        vertices = tri.simplices[simplex[valid]]

    return {
        "valid": valid,
        "vertices": vertices,
        "barycentric": barycentric,
        "n_target": len(dst_xy),
    }


def project(values: np.ndarray, interp: dict):
    values = np.asarray(values)
    one_dim = values.ndim == 1
    if one_dim:
        values = values[:, None]

    out = np.zeros((interp["n_target"], values.shape[1]), dtype=np.float32)
    if interp["valid"].any():
        gathered = values[interp["vertices"]]
        out[interp["valid"]] = np.sum(
            gathered * interp["barycentric"][:, :, None],
            axis=1,
        ).astype(np.float32)

    if one_dim:
        return out[:, 0]
    return out


def project_node_type(node_type: np.ndarray, src_xy: np.ndarray, dst_xy: np.ndarray):
    tree = cKDTree(src_xy)
    _, index = tree.query(dst_xy)
    return np.asarray(node_type[index], dtype=np.float32)


def output_dynamic_name(path: Path) -> str:
    stem = path.stem
    suffix = stem.split("_", 1)[1] if "_" in stem else stem
    return f"{DATASET_NAME}_{suffix}{path.suffix}"


In [ ]:
fine_mesh_path = require_path(FINE_MESH_SLF)
fine_base_path = require_path(FINE_BASE_BIN)
regular_mesh_path = require_path(REGULAR_MESH_SLF)
dynamic_paths = [require_path(path) for path in FINE_DYNAMIC_FILES if path]
if not dynamic_paths:
    raise ValueError("FINE_DYNAMIC_FILES is empty.")

output_dir = Path(OUTPUT_DIR).expanduser()
output_dir.mkdir(parents=True, exist_ok=True)

fine_mesh = TelemacFile(str(fine_mesh_path))
regular_mesh = TelemacFile(str(regular_mesh_path))
fine_graphs, _ = dgl.load_graphs(str(fine_base_path))
fine_base_graph = fine_graphs[0]

fine_xy, fine_triangles = add_mesh_info(fine_mesh)
regular_xy, regular_triangles = add_mesh_info(regular_mesh)
interp = build_interpolator(fine_xy, regular_xy)

fine_static = np.asarray(fine_base_graph.ndata["static"], dtype=np.float32)
fine_node_type = fine_static[:, :4]
fine_friction = fine_static[:, 4]
fine_bottom = fine_static[:, 5]

print("fine mesh:", fine_mesh_path)
print("fine base:", fine_base_path)
print("regular mesh:", regular_mesh_path)
print("output dir:", output_dir)
print("dynamic files:")
for path in dynamic_paths:
    print(" -", path)
print("fine nodes:", len(fine_xy), "fine triangles:", len(fine_triangles))
print("regular nodes:", len(regular_xy), "regular triangles:", len(regular_triangles))
print("valid projected points:", int(interp["valid"].sum()), "/", len(regular_xy))


In [ ]:
base_path = output_dir / f"{DATASET_NAME}_base.bin"
if base_path.exists() and not OVERWRITE:
    raise FileExistsError(base_path)

regular_node_type = project_node_type(fine_node_type, fine_xy, regular_xy)
regular_friction = project(fine_friction, interp)
regular_bottom = project(fine_bottom, interp)
regular_static = np.concatenate([
    regular_node_type,
    regular_friction[:, None],
    regular_bottom[:, None],
], axis=1).astype(np.float32)

regular_graph, regular_edge_features = get_dgl_graph(regular_mesh.tri)
regular_graph.edata["x"] = torch.as_tensor(regular_edge_features, dtype=torch.float32)
regular_graph.ndata["static"] = torch.as_tensor(regular_static, dtype=torch.float32)
dgl.save_graphs(str(base_path), [regular_graph])

print("base graph written:", base_path)
print("static shape:", tuple(regular_graph.ndata["static"].shape))


In [ ]:
for path in dynamic_paths:
    with open(path, "rb") as handle:
        dynamic_data = pickle.load(handle)

    projected_data = []
    for sample in dynamic_data:
        x, y, ts = unpack_dynamic_sample(sample)
        x_regular = project(x, interp)
        y_regular = project(y, interp)
        projected_data.append((x_regular, y_regular, ts))

    output_path = output_dir / output_dynamic_name(path)
    if output_path.exists() and not OVERWRITE:
        raise FileExistsError(output_path)

    with open(output_path, "wb") as handle:
        pickle.dump(projected_data, handle)

    print("written:", output_path, "| samples:", len(projected_data))
